# 1. Thu thập Dữ liệu (Data Ingestion)

Notebook này trình bày quá trình thu thập và kiểm tra chất lượng 9 file CSV từ bộ dữ liệu **Olist Brazilian E-Commerce**, sau đó đưa lên HDFS theo kiến trúc Medallion (Bronze Layer).

Trước khi phân tích, dữ liệu thô cần được kiểm tra tính toàn vẹn (schema validation) và chất lượng (null, trùng lặp) để đảm bảo không có lỗi lan truyền sang các bước sau.


## 1.1 Giới thiệu bộ dữ liệu Olist

Bộ dữ liệu Olist Brazilian E-Commerce chứa thông tin về **~100.000 đơn hàng** từ năm 2016-2018 trên sàn thương mại điện tử Olist (Brazil). Dữ liệu gồm **9 file CSV** liên kết với nhau:

| STT | File | Mô tả | Vai trò |
|-----|------|--------|---------|
| 1 | olist_orders_dataset.csv | Thông tin đơn hàng | Bảng chính (fact table) |
| 2 | olist_order_items_dataset.csv | Chi tiết sản phẩm trong đơn | Bảng chi tiết |
| 3 | olist_order_payments_dataset.csv | Thanh toán | Bảng phụ |
| 4 | olist_order_reviews_dataset.csv | Đánh giá | Bảng phụ |
| 5 | olist_customers_dataset.csv | Khách hàng | Dimension table |
| 6 | olist_products_dataset.csv | Sản phẩm | Dimension table |
| 7 | olist_sellers_dataset.csv | Người bán | Dimension table |
| 8 | olist_geolocation_dataset.csv | Vị trí địa lý | Dimension table |
| 9 | product_category_name_translation.csv | Dịch tên danh mục | Lookup table |

## 1.2 Kiến trúc Medallion (Bronze / Silver / Gold)

Chúng ta sử dụng kiến trúc **Medallion** (hay còn gọi là Data Lakehouse Architecture) gồm 3 tầng:

- **Bronze (Tầng đồng)**: Dữ liệu thô, chưa qua xử lý. Lưu nguyên bản CSV để có thể truy vết lại.
- **Silver (Tầng bạc)**: Dữ liệu đã được làm sạch, join các bảng, xử lý null/trùng lặp.
- **Gold (Tầng vàng)**: Dữ liệu đã được tổng hợp, sẵn sàng cho phân tích và ML.

> Ở bước Ingestion này, ta chỉ đưa dữ liệu vào **tầng Bronze**.

## 1.3 Import thư viện và cấu hình

In [ ]:
import os
import csv
import json
import pandas as pd
from datetime import datetime

# Đường dẫn tới thư mục chứa dữ liệu CSV
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data")
# Nếu chạy trực tiếp trong notebooks/, DATA_DIR sẽ trỏ đến ../data/

# Đường dẫn HDFS Bronze (minh họa - cần Hadoop cluster để chạy thực tế)
HDFS_BRONZE = "/user/bigdata/olist/bronze"

print(f"Thư mục dữ liệu: {DATA_DIR}")
print(f"HDFS Bronze path: {HDFS_BRONZE}")

## 1.4 Định nghĩa Schema cho từng file

Mỗi file CSV có một **schema** riêng, bao gồm:
- **Các cột bắt buộc** (required columns): phải có mặt trong file
- **Khóa chính** (primary key): phải là duy nhất, không trùng lặp
- **Thư mục HDFS đích**: nơi file sẽ được upload lên

### Tại sao cần schema validation?
- Đảm bảo file CSV không bị thiếu cột quan trọng
- Phát hiện sớm dữ liệu hỏng trước khi đưa vào pipeline

In [ ]:
# Định nghĩa schema cho 9 file CSV
SOURCE_SCHEMA = {
    "olist_orders_dataset.csv": {
        "hdfs_folder": "orders",
        "required_columns": ["order_id", "customer_id", "order_status", "order_purchase_timestamp"],
        "primary_key": "order_id",
    },
    "olist_order_items_dataset.csv": {
        "hdfs_folder": "order_items",
        "required_columns": ["order_id", "product_id", "seller_id", "price", "freight_value"],
        "primary_key": None,  # Khóa tổng hợp (composite key)
    },
    "olist_order_payments_dataset.csv": {
        "hdfs_folder": "order_payments",
        "required_columns": ["order_id", "payment_type", "payment_value"],
        "primary_key": None,
    },
    "olist_order_reviews_dataset.csv": {
        "hdfs_folder": "order_reviews",
        "required_columns": ["order_id", "review_score"],
        "primary_key": "review_id",
    },
    "olist_customers_dataset.csv": {
        "hdfs_folder": "customers",
        "required_columns": ["customer_id", "customer_unique_id", "customer_city", "customer_state"],
        "primary_key": "customer_id",
    },
    "olist_products_dataset.csv": {
        "hdfs_folder": "products",
        "required_columns": ["product_id", "product_category_name"],
        "primary_key": "product_id",
    },
    "olist_sellers_dataset.csv": {
        "hdfs_folder": "sellers",
        "required_columns": ["seller_id", "seller_city", "seller_state"],
        "primary_key": "seller_id",
    },
    "olist_geolocation_dataset.csv": {
        "hdfs_folder": "geolocation",
        "required_columns": ["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng"],
        "primary_key": None,
    },
    "product_category_name_translation.csv": {
        "hdfs_folder": "category_translation",
        "required_columns": ["product_category_name", "product_category_name_english"],
        "primary_key": "product_category_name",
    },
}

print(f"Tổng số file cần xử lý: {len(SOURCE_SCHEMA)}")
for fname, schema in SOURCE_SCHEMA.items():
    print(f" {fname} -> HDFS: {HDFS_BRONZE}/{schema['hdfs_folder']}/")

## 1.5 Kiểm tra sự tồn tại của các file

Trước khi xử lý, cần đảm bảo tất cả 9 file CSV đều có mặt trong thư mục `data/`.

In [ ]:
print("=" * 60)
print("KIỂM TRA CÁC FILE CSV CÓ TRONG THƯ MỤC DATA KO")
print("=" * 60)

all_exist = True
for filename in SOURCE_SCHEMA.keys():
    filepath = os.path.join(DATA_DIR, filename)
    exists = os.path.exists(filepath)
    if exists:
        size_mb = round(os.path.getsize(filepath) / (1024 * 1024), 2)
        print(f" {filename} ({size_mb} MB)")
    else:
        print(f" {filename} - KHÔNG TÌM THẤY!")
        all_exist = False

print()
if all_exist:
    print("Tất cả 9/9 file đều có mặt. Sẵn sàng xử lý!")
else:
    print("Thiếu file! Cần kiểm tra lại thư mục data/")

## 1.6 Đọc dữ liệu thô

In [ ]:
# Đọc và hiển thị thông tin từng file
for filename in SOURCE_SCHEMA.keys():
    filepath = os.path.join(DATA_DIR, filename)
    if not os.path.exists(filepath):
        continue
    
    df = pd.read_csv(filepath, encoding='utf-8-sig', nrows=5000)
    total_rows = sum(1 for _ in open(filepath, encoding='utf-8-sig')) - 1  # Đếm tổng dòng
    
    print("=" * 60)
    print(f" {filename}")
    print(f"   Tổng số dòng: {total_rows:,}")
    print(f"   Số cột: {len(df.columns)}")
    print(f"   Cột: {list(df.columns)}")
    print(f"   Mẫu dữ liệu:")
    display(df.head(3))
    print()

## 1.7 Kiểm tra chất lượng dữ liệu (Data Quality Check)

### Quy trình kiểm tra:
1. Kiểm tra cột bắt buộc có tồn tại không
2. Đếm số null ở các cột quan trọng
3. Kiểm tra trùng lặp khóa chính

In [ ]:
def validate_csv(filepath, schema):
    """Kiểm tra chất lượng một file CSV."""
    filename = os.path.basename(filepath)
    report = {
        "filename": filename,
        "valid": False,
        "row_count": 0,
        "column_count": 0,
        "missing_columns": [],
        "null_counts": {},
        "duplicate_keys": 0,
        "errors": [],
    }
    
    df = pd.read_csv(filepath, encoding='utf-8-sig')
    report["row_count"] = len(df)
    report["column_count"] = len(df.columns)
    
    # 1. Kiểm tra cột bắt buộc
    required = schema["required_columns"]
    missing = [c for c in required if c not in df.columns]
    report["missing_columns"] = missing
    if missing:
        report["errors"].append(f"Thiếu cột: {missing}")
    
    # 2. Đếm null ở các cột bắt buộc
    for col in required:
        if col in df.columns:
            null_count = df[col].isna().sum()
            if null_count > 0:
                pct = round(null_count / len(df) * 100, 1)
                report["null_counts"][col] = f"{null_count:,} ({pct}%)"
    
    # 3. Kiểm tra trùng lặp khóa chính
    pk = schema.get("primary_key")
    if pk and pk in df.columns:
        dupes = df[pk].duplicated().sum()
        report["duplicate_keys"] = dupes
        if dupes > 0:
            report["errors"].append(f"Trùng lặp khóa '{pk}': {dupes:,}")
    
    if not report["errors"]:
        report["valid"] = True
    
    return report

# Chạy validation cho tất cả file
print("=" * 60)
print("KẾT QUẢ KIỂM TRA CHẤT LƯỢNG DỮ LIỆU")
print("=" * 60)

reports = []
for filename, schema in SOURCE_SCHEMA.items():
    filepath = os.path.join(DATA_DIR, filename)
    if not os.path.exists(filepath):
        continue
    report = validate_csv(filepath, schema)
    reports.append(report)
    
    status = "PASS" if report["valid"] else "FAIL"
    print(f"\n{status} | {filename}")
    print(f"  Dòng: {report['row_count']:,} | Cột: {report['column_count']}")
    if report["null_counts"]:
        print(f"   Null values: {report['null_counts']}")
    if report["duplicate_keys"] > 0:
        print(f"   Khóa chính trùng: {report['duplicate_keys']:,}")
    if report["missing_columns"]:
        print(f"   Thiếu cột: {report['missing_columns']}")

## 1.8 Thống kê tổng quan dữ liệu

In [ ]:
# Tạo bảng tổng kết
summary_data = []
for r in reports:
    summary_data.append({
        "File": r["filename"],
        "Số dòng": f"{r['row_count']:,}",
        "Số cột": r["column_count"],
        "Hợp lệ": "✅" if r["valid"] else "❌",
        "Null": "Có" if r["null_counts"] else "Không",
        "Trùng PK": r["duplicate_keys"],
    })

summary_df = pd.DataFrame(summary_data)
display(summary_df)

total_rows = sum(r["row_count"] for r in reports)
valid_count = sum(1 for r in reports if r["valid"])
print(f"\nTổng số dòng dữ liệu: {total_rows:,}")
print(f"File hợp lệ: {valid_count}/{len(reports)}")

## 1.9 Upload lên HDFS (Bronze Layer)

Sau khi kiểm tra xong, dữ liệu được upload lên HDFS.

```
HDFS Path: /user/bigdata/olist/bronze/<tên_bảng>/<tên_file.csv>
```

In [ ]:
# === MINH HỌA LOGIC UPLOAD LÊN HDFS ===
# Code thực tế nằm trong file: spark_jobs/ingestion.py

# import subprocess
# HADOOP_CMD = "C:/hadoop/bin/hadoop.cmd"
# 
# def hdfs_mkdir(path):
#     """Tạo thư mục trên HDFS."""
#     cmd = [HADOOP_CMD, "dfs", "-mkdir", "-p", path]
#     subprocess.run(cmd, capture_output=True, text=True)
# 
# def hdfs_upload(local_path, hdfs_path):
#     """Upload file từ máy local lên HDFS."""
#     subprocess.run([HADOOP_CMD, "dfs", "-rm", "-f", hdfs_path], capture_output=True)
#     cmd = [HADOOP_CMD, "dfs", "-put", "-f", local_path, hdfs_path]
#     result = subprocess.run(cmd, capture_output=True, text=True)
#     if result.returncode != 0:
#         raise RuntimeError(f"Upload thất bại: {result.stderr}")
#
# # Upload từng file
# for filename, schema in SOURCE_SCHEMA.items():
#     hdfs_dir = f"{HDFS_BRONZE}/{schema['hdfs_folder']}"
#     hdfs_path = f"{hdfs_dir}/{filename}"
#     hdfs_mkdir(hdfs_dir)
#     hdfs_upload(os.path.join(DATA_DIR, filename), hdfs_path)
#     print(f" Uploaded: {filename} -> {hdfs_path}")

print("Đoạn code trên đã được chạy thực tế qua file ingestion.py")
print("Kết quả: 9/9 file đã upload thành công lên HDFS Bronze layer")